# Group 10 - Full Pipeline Notebook

| Cell | What happens |
|------|-------------|
| 1 | Data preparation|
| 2 | Hyperparameter tuning| 
| 3 | Train V1 baseline|
| 4 | Train V2 snow model|
| 5 | Evaluate + compare |

## Step 0 — Load config (everyone uses this)

In [ ]:
from src.common_utils import load_config
from src.common_utils.seed import set_seed_from_config
from src.data import DataPipeline
from src.model.baseline_yolov9.trainer import YOLOv9Trainer
from src.model.snow_augmented_yolov9.trainer_sa import YOLOv9TrainerSA
from src.model.evaluate import Evaluator

print("All modules imported successfully!")

In [ ]:
from src.common_utils import load_config
from src.common_utils.seed import set_seed_from_config

cfg = load_config()          # reads config.yaml, active variant = v1 by default
set_seed_from_config(cfg)    # fixes random seed = 42 for reproducibility

print('Config loaded.')
print(f'  device : {cfg.project.device}')
print(f'  variant: {cfg.variants.active}')
print(f'  epochs : {cfg.training.epochs}')
print(f'  lr     : {cfg.training.lr}')

## Step 1 — Data pipeline 

In [ ]:
from src.data import DataPipeline

pipeline = DataPipeline('config.yaml')
# First run: processes videos → slow (minutes)
# Every run after: skips setup, instant
pipeline.summary()

## Step 2 — Hyperparameter tuning 
Skip this cell if tuning already ran — best values are already in config.yaml.

In [ ]:
SKIP_TUNING = True   # ← set False to run tuning (takes ~2 hours)

if not SKIP_TUNING:
    from src.tuning import run_study, write_best_to_config
    study = run_study()             # Sathish's Optuna code
    write_best_to_config(study)     # writes best lr, batch_size etc into config.yaml
    cfg = load_config()             # reload config with updated values
    print('Tuning done. config.yaml updated.')
else:
    print('Skipping tuning — using values already in config.yaml')
    print(f'  lr         : {cfg.training.lr}')
    print(f'  batch_size : {cfg.training.batch_size}')
    print(f'  box_weight : {cfg.loss.box_weight}')

## Step 3 — Train V1 baseline 

In [ ]:
from src.model.baseline_yolov9.trainer import YOLOv9Trainer

cfg_v1     = load_config(variant='v1')
# cfg_v1.training.lr      ← whatever is now in config.yaml (default or tuned)
# cfg_v1.training.epochs  ← 100
# cfg_v1.use_snow_aug     ← False

trainer_v1 = YOLOv9Trainer(cfg_v1)
trainer_v1.train()
# Saves → outputs/checkpoints/v1/best.pt

## Step 4 — Train V2 snow model

In [ ]:
from src.model.snow_augmented_yolov9.trainer_sa import YOLOv9TrainerSA

cfg_v2     = load_config(variant='v2')
# cfg_v2.training.lr      ← same tuned values
# cfg_v2.training.epochs  ← 120
# cfg_v2.use_snow_aug     ← True

trainer_v2 = YOLOv9TrainerSA(cfg_v2)
trainer_v2.train()
# Saves → outputs/checkpoints/v2/best.pt

## Step 5 — Evaluate and compare

In [ ]:
from src.model.evaluate import Evaluator

# Runs both models on the test set and prints side-by-side results
Evaluator.compare('v1', 'v2')

# These numbers go straight into your presentation slides